In [22]:
import os
import math
import pandas as pd

from typing import Optional, Tuple
from dotenv import load_dotenv
from PIL import Image

from smolagents import (
    CodeAgent,
    LiteLLMModel,
    GoogleSearchTool,
    VisitWebpageTool,
    tool,
)

load_dotenv()

True

In [23]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL", "groq/openai/gpt-oss-120b")

model = LiteLLMModel(
    model_id=GROQ_MODEL,
    api_key=GROQ_API_KEY,
    temperature=0.2,
    max_tokens=4096,
)

In [24]:
@tool
def calculate_cargo_travel_time(
    origin_coords: Tuple[float, float],
    destination_coords: Tuple[float, float],
    cruising_speed_kmh: Optional[float] = 750.0,
) -> float:
    """
    Calculate the travel time for a cargo plane between two points on Earth.

    Args:
        origin_coords: Tuple of (latitude, longitude) for starting location.
        destination_coords: Tuple of (latitude, longitude) for destination.
        cruising_speed_kmh: Cargo plane cruising speed in km/h.

    Returns:
        Estimated cargo plane travel time in hours.
    """

    def to_radians(degrees: float) -> float:
        return degrees * (math.pi / 180)

    # Extract coordinates
    lat1, lon1 = map(to_radians, origin_coords)
    lat2, lon2 = map(to_radians, destination_coords)

    # Earth's radius
    EARTH_RADIUS_KM = 6371.0

    # Calculate coordinate differences
    dlon = lon2 - lon1
    dlat = lat2 - lat1

    # Haversine formula
    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    )

    c = 2 * math.asin(math.sqrt(a))

    distance = EARTH_RADIUS_KM * c

    # Account for non-direct routes
    actual_distance = distance * 1.1

    # Calculate flight time
    flight_time = (actual_distance / cruising_speed_kmh) + 1.0

    return round(flight_time, 2)

In [25]:
print(
    calculate_cargo_travel_time(
        (41.8781, -87.6298),
        (-33.8688, 151.2093)
    )
)

22.82


In [26]:
task = """
Find Batman filming locations around the world.

For each filming location:

1. Find its geographical coordinates (latitude and longitude).
2. Calculate the cargo plane travel time from that location to Gotham.
3. Use Gotham's coordinates:
   Latitude: 40.7128
   Longitude: -74.0060

Also identify some supercar manufacturing factories around the world.

For each factory:

1. Find its geographical coordinates.
2. Calculate the cargo plane transfer time to Gotham.
3. Include the factory name and associated car manufacturer.

Return the results as two separate pandas DataFrames:

1. Batman Filming Locations
2. Supercar Factories

Include relevant sources wherever possible.
"""

In [29]:
agent = CodeAgent(
    model=model,
    tools=[
        # GoogleSearchTool(),
        VisitWebpageTool(),
        calculate_cargo_travel_time,
    ],
    additional_authorized_imports=["pandas"],
    max_steps=5,
    verbosity_level=1,
)

In [30]:
result = agent.run(task)

print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find Batman filming locations around the world.                                                                 │
│                                                                                                                 │
│ For each filming location:                                                                                      │
│                                                                                                                 │
│ 1. Find its geographical coordinates (latitude and longitude).                                                  │
│ 2. Calculate the cargo plane travel time from that location to Gotham.                                          │
│ 3. Use Gotham's coordinates:                                                                                    │
│    Latitude: 40.7128                                                                                            │
│    Longitude: -74.0060                                                                                          │
│                                                                                                                 │
│ Also identify some supercar manufacturing factories around the world.                                           │
│                                                                                                                 │
│ For each factory:                                                                                               │
│                                                                                                                 │
│ 1. Find its geographical coordinates.                                                                           │
│ 2. Calculate the cargo plane transfer time to Gotham.                                                           │
│ 3. Include the factory name and associated car manufacturer.                                                    │
│                                                                                                                 │
│ Return the results as two separate pandas DataFrames:                                                           │
│                                                                                                                 │
│ 1. Batman Filming Locations                                                                                     │
│ 2. Supercar Factories                                                                                           │
│                                                                                                                 │
│ Include relevant sources wherever possible.                                                                     │
│                                                                                                                 │
╰─ LiteLLMModel - groq/openai/gpt-oss-120b ───────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import pandas as pd                                                                                              
                                                                                                                   
  # ----------------------------------------------------------------------                                         
  # 1. Define Gotham coordinates (NYC) and cruising speed for cargo planes                                         
  # ----------------------------------------------------------------------                                         
  gotham_coords = (40.7128, -74.0060)          # (lat, lon)                                                        
  cruising_speed_kmh = 800.0                  # typical cargo jet speed                                            
                                                                                                                   
  # ----------------------------------------------------------------------                                         
  # 2. Batman filming locations (city, country, lat, lon, source)                                                  
  # ----------------------------------------------------------------------                                         
  batman_locations = [                                                                                             
      {                                                                                                            
          "Location": "Chicago, Illinois, USA",                                                                    
          "Latitude": 41.8781,                                                                                     
          "Longitude": -87.6298,                                                                                   
          "Source": "https://en.wikipedia.org/wiki/Chicago"                                                        
      },                                                                                                           
      {                                                                                                            
          "Location": "London, England, UK",                                                                       
          "Latitude": 51.5074,                                                                                     
          "Longitude": -0.1278,                                                                                    
          "Source": "https://en.wikipedia.org/wiki/London"                                                         
      },                                                                                                           
      {                                                                                                            
          "Location": "Hong Kong, China",                                                                          
          "Latitude": 22.3193,                                                                                     
          "Longitude": 114.1694,                                                                                   
          "Source": "https://en.wikipedia.org/wiki/Hong_Kong"                                                      
      },                                                                                                           
      {                                                                                                            
          "Location": "Reykjavik, Iceland",                                                                        
          "Latitude": 64.1466,                                                                                     
          "Longitude": -21.9426,                         

Final answer: {'Batman Filming Locations':                        Location  Latitude  Longitude  Travel_Time_h  \
0        Chicago, Illinois, USA   41.8781   -87.6298           2.57   
1           London, England, UK   51.5074    -0.1278           8.66   
2              Hong Kong, China   22.3193   114.1694          18.81   
3            Reykjavik, Iceland   64.1466   -21.9426           6.78   
4  Los Angeles, California, USA   34.0522  -118.2437           6.41   
5         Glasgow, Scotland, UK   55.8642    -4.2518           8.12   

                                      Source  
0      https://en.wikipedia.org/wiki/Chicago  
1       https://en.wikipedia.org/wiki/London  
2    https://en.wikipedia.org/wiki/Hong_Kong  
3    https://en.wikipedia.org/wiki/Reykjavik  
4  https://en.wikipedia.org/wiki/Los_Angeles  
5      https://en.wikipedia.org/wiki/Glasgow  , 'Supercar Factories':                                   Factory    
Manufacturer  Latitude  \
0                 Ferrari Maranello Plant             Ferrari   44.5400   
1  Lamborghini Sant'Agata Bolognese Plant         Lamborghini   44.5600   
2                  Bugatti Molsheim Plant             Bugatti   48.5400   
3               McLaren Technology Centre  McLaren Automotive   51.2986   
4                   Porsche Leipzig Plant             Porsche   51.3400   
5          Koenigsegg Production Facility          Koenigsegg   56.2850   
6                          Pagani Factory              Pagani   44.5600   
7                  Rimac Automobili Plant               Rimac   45.8000   

   Longitude  Travel_Time_h                                             Source  
0    10.8770          10.12              https://en.wikipedia.org/wiki/Ferrari  
1    11.2500          10.15          https://en.wikipedia.org/wiki/Lamborghini  
2     7.4600           9.52              https://en.wikipedia.org/wiki/Bugatti  
3    -0.5605           8.63  https://en.wikipedia.org/wiki/McLaren_Technolo...  
4    12.3800           9.77      https://en.wikipedia.org/wiki/Porsche_Leipzig  
5    12.8620           9.49           https://en.wikipedia.org/wiki/Koenigsegg  
6    10.9500          10.12               https://en.wikipedia.org/wiki/Pagani  
7    15.9500          10.49     https://en.wikipedia.org/wiki/Rimac_Automobili  }

[Step 1: Duration 6.03 seconds| Input tokens: 2,315 | Output tokens: 2,104]

{'Batman Filming Locations':                        Location  Latitude  Longitude  Travel_Time_h  \
0        Chicago, Illinois, USA   41.8781   -87.6298           2.57   
1           London, England, UK   51.5074    -0.1278           8.66   
2              Hong Kong, China   22.3193   114.1694          18.81   
3            Reykjavik, Iceland   64.1466   -21.9426           6.78   
4  Los Angeles, California, USA   34.0522  -118.2437           6.41   
5         Glasgow, Scotland, UK   55.8642    -4.2518           8.12   

                                      Source  
0      https://en.wikipedia.org/wiki/Chicago  
1       https://en.wikipedia.org/wiki/London  
2    https://en.wikipedia.org/wiki/Hong_Kong  
3    https://en.wikipedia.org/wiki/Reykjavik  
4  https://en.wikipedia.org/wiki/Los_Angeles  
5      https://en.wikipedia.org/wiki/Glasgow  , 'Supercar Factories':                                   Factory        Manufacturer  Latitude  \
0                 Ferrari Maranello Plant     

In [31]:
result

{'Batman Filming Locations':                        Location  Latitude  Longitude  Travel_Time_h  \
 0        Chicago, Illinois, USA   41.8781   -87.6298           2.57   
 1           London, England, UK   51.5074    -0.1278           8.66   
 2              Hong Kong, China   22.3193   114.1694          18.81   
 3            Reykjavik, Iceland   64.1466   -21.9426           6.78   
 4  Los Angeles, California, USA   34.0522  -118.2437           6.41   
 5         Glasgow, Scotland, UK   55.8642    -4.2518           8.12   
 
                                       Source  
 0      https://en.wikipedia.org/wiki/Chicago  
 1       https://en.wikipedia.org/wiki/London  
 2    https://en.wikipedia.org/wiki/Hong_Kong  
 3    https://en.wikipedia.org/wiki/Reykjavik  
 4  https://en.wikipedia.org/wiki/Los_Angeles  
 5      https://en.wikipedia.org/wiki/Glasgow  ,
 'Supercar Factories':                                   Factory        Manufacturer  Latitude  \
 0                 Ferrari Mara